# Bob Portfolio 자동 갱신

Notion `매매일지` → `보유주식` → `총자산` UPSERT 자동화.

## 사전 준비
1. https://www.notion.so/profile/integrations 접속 → `New integration` → Internal type
2. Capabilities: Read/Update/Insert content 모두 체크
3. Secret token 복사 (`ntn_...` 형식)
4. Bob Portfolio 페이지 우상단 `•••` → `Connect to` → 만든 integration 선택
5. 이 노트북 Colab Secrets 패널에서 `NOTION_TOKEN` 키로 저장

## 동작 순서
1. 매매일지 fetch → 티커별 집계 (수량/가중평균매입가)
2. 보유주식 UPSERT (티커 기준)
3. 시세 fetch (국내=pykrx, 해외=yfinance)
4. 보유주식 평가금액/수익/수익률 갱신
5. 총자산 합계 UPSERT (작성일자 기준)

## 절대 규칙
- 노션 페이지 블록 삭제 금지. row UPSERT만 허용.

In [ ]:
!pip install -q requests pandas pykrx yfinance

## 1. 설정

In [ ]:
from google.colab import userdata
import requests, pandas as pd
from datetime import date, timedelta
from collections import defaultdict
from pykrx import stock
import yfinance as yf

NOTION_TOKEN = userdata.get('NOTION_TOKEN')
NOTION_VERSION = '2025-09-03'

DS_TRADE   = 'dae5f91b-fdbe-4787-90d9-f6966a727a2c'  # 매매일지
DS_HOLDING = 'ae611b21-f20d-438a-a624-e81f34909de4'  # 보유주식
DS_ASSET   = '77ab4124-838b-4ded-bef9-f10f337a86ec'  # 총자산

H = {
    'Authorization': f'Bearer {NOTION_TOKEN}',
    'Notion-Version': NOTION_VERSION,
    'Content-Type': 'application/json',
}

## 2. Notion API 헬퍼

In [ ]:
API = 'https://api.notion.com/v1'

def query_ds(ds_id, filter_=None, sorts=None):
    out, cursor = [], None
    while True:
        body = {'page_size': 100}
        if cursor: body['start_cursor'] = cursor
        if filter_: body['filter'] = filter_
        if sorts: body['sorts'] = sorts
        r = requests.post(f'{API}/data_sources/{ds_id}/query', headers=H, json=body)
        r.raise_for_status()
        d = r.json()
        out.extend(d['results'])
        if not d.get('has_more'): break
        cursor = d['next_cursor']
    return out

def create_page(ds_id, props):
    r = requests.post(f'{API}/pages', headers=H, json={
        'parent': {'type': 'data_source_id', 'data_source_id': ds_id},
        'properties': props,
    })
    r.raise_for_status()
    return r.json()

def update_page(page_id, props):
    r = requests.patch(f'{API}/pages/{page_id}', headers=H, json={'properties': props})
    r.raise_for_status()
    return r.json()

def title(s):       return {'title': [{'text': {'content': str(s)}}]}
def rich(s):        return {'rich_text': [{'text': {'content': str(s)}}]}
def num(v):         return {'number': float(v) if v is not None else None}
def select(name):   return {'select': {'name': name}} if name else {'select': None}
def date_(d):       return {'date': {'start': str(d)}} if d else {'date': None}

def get_text(prop):
    if not prop: return ''
    t = prop.get('type')
    if t == 'title':       return ''.join(x['plain_text'] for x in prop['title'])
    if t == 'rich_text':   return ''.join(x['plain_text'] for x in prop['rich_text'])
    if t == 'select':      return prop['select']['name'] if prop['select'] else ''
    if t == 'number':      return prop['number']
    if t == 'date':        return prop['date']['start'] if prop['date'] else ''
    return ''

## 2.5 진단 (404 / 401 디버깅용)

오류 발생 시 이 셀로 원인 좁히기.
- [1] 401 → 토큰 오타 또는 만료
- [2] 404 → integration이 Bob Portfolio 페이지에 connect 안 됨. 페이지 우상단 `⋮` → `Connections` → integration 추가

In [ ]:
r = requests.get(f'{API}/users/me', headers=H)
print('[1] users/me:', r.status_code, r.json().get('name') if r.ok else r.text[:200])

for label, ds in [('TRADE', DS_TRADE), ('HOLDING', DS_HOLDING), ('ASSET', DS_ASSET)]:
    r = requests.get(f'{API}/data_sources/{ds}', headers=H)
    print(f'[2] {label} {ds}:', r.status_code)
    if not r.ok:
        print('   →', r.text[:300])

## 3. 매매일지 → 보유주식 집계

티커별로 그룹핑. 가중평균 매입가 = Σ(매수수량 × 단가) / Σ매수수량. 매도는 보유수량만 차감 (간단화: 평균매입가는 유지).

In [ ]:
trades = query_ds(DS_TRADE)
agg = defaultdict(lambda: {'name': '', 'category': '', 'qty': 0.0, 'buy_qty': 0.0, 'buy_amt': 0.0})
for p in trades:
    pp = p['properties']
    ticker = get_text(pp.get('티커'))
    if not ticker: continue
    side = get_text(pp.get('매수/매도'))
    qty  = float(get_text(pp.get('수량')) or 0)
    price = float(get_text(pp.get('단가')) or 0)
    a = agg[ticker]
    a['name']     = get_text(pp.get('종목이름'))
    a['category'] = get_text(pp.get('분류'))
    if side == '매수':
        a['qty']     += qty
        a['buy_qty'] += qty
        a['buy_amt'] += qty * price
    elif side == '매도':
        a['qty']     -= qty

holdings = {}
for ticker, a in agg.items():
    if a['qty'] <= 0: continue
    avg = a['buy_amt'] / a['buy_qty'] if a['buy_qty'] else 0
    holdings[ticker] = {**a, 'avg_price': avg}

pd.DataFrame(holdings).T

## 4. 시세 fetch

- 국내(`국내종목`/`국내ETF`/`국내ETF-해외`): pykrx 종가
- 해외(`해외종목`/`해외ETF`): yfinance

In [ ]:
def last_krx_price(ticker):
    today = date.today()
    start = (today - timedelta(days=14)).strftime('%Y%m%d')
    end   = today.strftime('%Y%m%d')
    df = stock.get_market_ohlcv_by_date(start, end, ticker)
    if df.empty: return None
    return float(df['종가'].iloc[-1])

def last_yf_price(ticker):
    t = yf.Ticker(ticker)
    h = t.history(period='5d')
    if h.empty: return None
    return float(h['Close'].iloc[-1])

KRX_CATS = {'국내종목', '국내ETF', '국내ETF-해외'}

for ticker, h in holdings.items():
    try:
        price = last_krx_price(ticker) if h['category'] in KRX_CATS else last_yf_price(ticker)
    except Exception as e:
        print(f'{ticker} 시세 실패: {e}'); price = None
    h['current'] = price or h['avg_price']
    h['valuation'] = h['qty'] * h['current']
    h['profit']    = (h['current'] - h['avg_price']) * h['qty']
    h['profit_rate'] = h['profit'] / (h['avg_price'] * h['qty']) if h['avg_price'] else 0

pd.DataFrame(holdings).T

## 5. 보유주식 UPSERT

티커 일치 row 찾으면 update, 없으면 create. 페이지 자체는 절대 삭제 안 함.

In [ ]:
existing = query_ds(DS_HOLDING)
by_ticker = {get_text(p['properties'].get('티커')): p['id'] for p in existing}

for ticker, h in holdings.items():
    props = {
        '종목이름': title(h['name']),
        '티커':     rich(ticker),
        '보유수량': num(h['qty']),
        '매입가':   num(h['avg_price']),
        '평가금액': num(h['valuation']),
        '수익':     num(h['profit']),
        '수익률':   num(h['profit_rate']),
        '분류':     select(h['category']) if h['category'] else None,
    }
    props = {k: v for k, v in props.items() if v is not None}
    if ticker in by_ticker:
        update_page(by_ticker[ticker], props)
        print(f'UPDATE {ticker} {h["name"]}')
    else:
        create_page(DS_HOLDING, props)
        print(f'CREATE {ticker} {h["name"]}')

## 6. 총자산 UPSERT

작성일자(오늘) 기준. 같은 날 row 있으면 update.

In [ ]:
total_val = sum(h['valuation'] for h in holdings.values())
total_buy = sum(h['avg_price'] * h['qty'] for h in holdings.values())
total_pl  = total_val - total_buy
total_pr  = total_pl / total_buy if total_buy else 0
today_str = date.today().isoformat()

asset_rows = query_ds(DS_ASSET)
today_id = None
for p in asset_rows:
    if get_text(p['properties'].get('작성일자')) == today_str:
        today_id = p['id']; break

props = {
    '작성일자':   title(today_str),
    '총평가금액': num(total_val),
    '총수익':     num(total_pl),
    '총수익률':   num(total_pr),
}
if today_id:
    update_page(today_id, props)
    print(f'UPDATE 총자산 {today_str} ₩{int(total_val):,}')
else:
    create_page(DS_ASSET, props)
    print(f'CREATE 총자산 {today_str} ₩{int(total_val):,}')

## 7. 분류별 비율 파이차트

보유주식 분류(국내종목/국내ETF/국내ETF-해외/해외종목/해외ETF)별 평가금액 비율 파이차트.
Imgur 업로드 후 Notion 이미지 블록 URL 자동 갱신.

**사전 준비**:
1. https://api.imgur.com/oauth2/addclient → `New Application` 등록 (`Anonymous usage without user authorization` 선택)
2. Client-ID 복사
3. Colab Secrets 패널에서 `IMGUR_CLIENT_ID` 키로 저장

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import subprocess

subprocess.run(['apt-get', '-y', 'install', 'fonts-nanum'], capture_output=True)
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

cat_total = defaultdict(float)
for h in holdings.values():
    cat_total[h['category'] or '미분류'] += h['valuation']

labels = list(cat_total.keys())
sizes  = list(cat_total.values())
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974']

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, autopct='%1.1f%%',
    startangle=90, colors=colors[:len(labels)],
    textprops={'fontsize': 13}
)
for at in autotexts:
    at.set_color('white'); at.set_fontweight('bold')
ax.set_title(f'보유주식 분류별 비율 ({date.today().isoformat()})', fontsize=16, pad=20)
plt.tight_layout()
plt.savefig('pie.png', bbox_inches='tight', dpi=120, facecolor='white')
plt.show()

for label, size in zip(labels, sizes):
    print(f'  {label}: ₩{int(size):,} ({size/sum(sizes)*100:.1f}%)')

In [ ]:
import base64

IMGUR_CLIENT_ID = userdata.get('IMGUR_CLIENT_ID')

with open('pie.png', 'rb') as f:
    img_b64 = base64.b64encode(f.read())

r = requests.post(
    'https://api.imgur.com/3/image',
    headers={'Authorization': f'Client-ID {IMGUR_CLIENT_ID}'},
    data={'image': img_b64, 'type': 'base64'},
)
r.raise_for_status()
img_url = r.json()['data']['link']
print('Imgur URL:', img_url)

In [ ]:
PAGE_ID = '380940b8-828c-81a0-88f6-eadcff08e9c6'

def list_children(block_id):
    out, cursor = [], None
    while True:
        params = {'page_size': 100}
        if cursor: params['start_cursor'] = cursor
        r = requests.get(f'{API}/blocks/{block_id}/children', headers=H, params=params)
        r.raise_for_status()
        d = r.json()
        out.extend(d['results'])
        if not d.get('has_more'): break
        cursor = d['next_cursor']
    return out

children = list_children(PAGE_ID)
target_block_id = None
after_heading = False
for b in children:
    if b['type'] == 'heading_2':
        text = ''.join(t['plain_text'] for t in b['heading_2']['rich_text'])
        after_heading = '분류별 비율' in text
    elif after_heading and b['type'] == 'image':
        target_block_id = b['id']; break

if not target_block_id:
    raise RuntimeError('"분류별 비율" 헤딩 다음 image 블록 없음. 노션 페이지 확인.')

r = requests.patch(
    f'{API}/blocks/{target_block_id}',
    headers=H,
    json={'image': {'external': {'url': img_url}}},
)
r.raise_for_status()
print(f'이미지 블록 갱신: {target_block_id} → {img_url}')

## 완료

Bob Portfolio 페이지 새로고침하면 갱신된 데이터 반영. 매일 1회 실행 권장.